# 02 Labeling Dataset — Соусы

Цель: из `research/dedup/data/candidates_sauces.csv` собрать компактный stratified batch для ручной разметки gold-set. Ноутбук не генерирует `label`: человек заполняет эту колонку вручную.


## Инструкция по ручной разметке

Заполняйте колонку `label` одним из значений: `exact_duplicate`, `same_product_different_pack`, `different_product`, `uncertain`. В `notes` можно кратко объяснить сомнение или правило.

- `exact_duplicate`: один и тот же товар в той же фасовке, просто разные карточки/артикулы. Пример из EDA/candidates: `Набор из трех трюфельных соусов Tris: ... Giuliano Tartufi, Италия` vs тот же title, brand `giuliano tartufi`, вес `0.08`/`0.08`.
- `same_product_different_pack`: та же базовая позиция, но другой pack/multipack; по бизнес-правилу это не auto-merge в один SKU. Пример: `Соус соевой, 500 мл` vs `Соус соевой, 500 мл - 2 шт`, brand `обок`, unit `0.5`, total `0.5` vs `1.0`.
- `different_product`: похожая карточка того же brand/веса, но другой вкус/тип продукта. Пример hard-negative из EDA/candidates: `Соус Барбекю "Ноль грамм", 330г для мяса...` vs `Соус Сладкий чили "Ноль грамм" 330г...`, тот же вес, но разные flavor-токены.
- `uncertain`: данных недостаточно или пара спорная без просмотра карточки/состава. Пример: `Горчица Баварская HAAS 1 кг.` vs `Горчица Haas Дижонская, 3 шт по 170 г` — похожий brand, но одновременно меняются тип и pack.

`labeling_stratum`, `is_cross_marketplace_pair`, `is_hard_negative_candidate` и `is_pack_variant_candidate` — подсказки для отбора, а не правильный ответ.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import (
    LabelingSamplingConfig,
    add_cross_marketplace_flags,
    add_pack_variant_flags,
    stratified_labeling_sample,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 140)


In [ ]:
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
CANDIDATES_PATH = DATA_DIR / "candidates_sauces.csv"
LABELING_PATH = DATA_DIR / "labeling_sauces.csv"

TARGET_SAMPLE_SIZE = int(os.environ.get("DEDUP_LABELING_TARGET_SIZE", "400"))
RANDOM_STATE = int(os.environ.get("DEDUP_LABELING_RANDOM_STATE", "42"))

sampling_config = LabelingSamplingConfig(
    target_size=TARGET_SAMPLE_SIZE,
    random_state=RANDOM_STATE,
    high_similarity_threshold=0.72,
    medium_similarity_lower=0.50,
    medium_similarity_upper=0.72,
    easy_negative_upper=0.50,
)

print(f"Target sample size: {sampling_config.target_size}")
print(f"Random state: {sampling_config.random_state}")


In [ ]:
if not CANDIDATES_PATH.exists():
    raise FileNotFoundError(
        f"Не найден {CANDIDATES_PATH}. Сначала выполните notebooks/01_candidate_generation.ipynb."
    )

candidates = pd.read_csv(CANDIDATES_PATH)
candidates = add_cross_marketplace_flags(add_pack_variant_flags(candidates))
print(f"Loaded candidates: {len(candidates):,}")
display(candidates.head(5))


In [ ]:
score = pd.to_numeric(candidates["baseline_similarity_score"], errors="coerce").fillna(0.0)
availability = pd.DataFrame(
    [
        {
            "stratum": "cross_marketplace_candidate",
            "available_pairs": int(candidates["is_cross_marketplace_pair"].fillna(False).sum()),
        },
        {"stratum": "hard_negative_candidate", "available_pairs": int(candidates["is_hard_negative_candidate"].fillna(False).sum())},
        {"stratum": "pack_variant_candidate", "available_pairs": int(candidates["is_pack_variant_candidate"].fillna(False).sum())},
        {"stratum": "high_similarity", "available_pairs": int((score >= sampling_config.high_similarity_threshold).sum())},
        {
            "stratum": "medium_similarity",
            "available_pairs": int(((score >= sampling_config.medium_similarity_lower) & (score < sampling_config.medium_similarity_upper)).sum()),
        },
        {"stratum": "random_easy_negative", "available_pairs": int((score < sampling_config.easy_negative_upper).sum())},
    ]
)
display(availability)


In [ ]:
labeling_df = stratified_labeling_sample(candidates, sampling_config)

export_columns = [
    "label",
    "notes",
    "labeling_stratum",
    "raw_record_id_a",
    "raw_record_id_b",
    "marketplace_a",
    "marketplace_b",
    "marketplaces_a",
    "marketplaces_b",
    "sku_a",
    "sku_b",
    "title_a",
    "title_b",
    "brand_a",
    "brand_b",
    "unit_amount_a",
    "unit_amount_b",
    "total_amount_a",
    "total_amount_b",
    "multipack_count_a",
    "multipack_count_b",
    "baseline_similarity_score",
    "is_cross_marketplace_pair",
    "is_hard_negative_candidate",
    "is_pack_variant_candidate",
]
labeling_df = labeling_df[export_columns].copy()

DATA_DIR.mkdir(parents=True, exist_ok=True)
labeling_df.to_csv(LABELING_PATH, index=False)
print(f"Saved labeling dataset: {LABELING_PATH}")
print(f"Rows saved: {len(labeling_df):,}")


In [ ]:
strata_stats = (
    labeling_df.groupby("labeling_stratum")
    .agg(
        pairs=("labeling_stratum", "size"),
        cross_marketplace_pairs=("is_cross_marketplace_pair", "sum"),
    )
    .reset_index()
    .sort_values("labeling_stratum")
)
display(strata_stats)

score_by_stratum = labeling_df.groupby("labeling_stratum")["baseline_similarity_score"].agg(
    pairs="count",
    min="min",
    median="median",
    max="max",
).reset_index()
display(score_by_stratum)

cross_marketplace_summary = pd.DataFrame(
    [
        {
            "rows_total": len(labeling_df),
            "cross_marketplace_pairs": int(labeling_df["is_cross_marketplace_pair"].fillna(False).sum()),
            "same_marketplace_pairs": int((~labeling_df["is_cross_marketplace_pair"].fillna(False)).sum()),
        }
    ]
)
display(cross_marketplace_summary)


## Sanity-check before manual work

Проверяем, что экспорт не содержит автоматически заполненных labels и что каждая страта выглядит как ожидаемый тип ручного контроля.


In [ ]:
assert labeling_df["label"].fillna("").eq("").all(), "label должен оставаться пустым для ручной разметки"
assert labeling_df["notes"].fillna("").eq("").all(), "notes должен оставаться пустым для ручной разметки"
assert len(labeling_df) <= sampling_config.target_size

for stratum in strata_stats["labeling_stratum"].tolist():
    print(stratum)
    display(
        labeling_df[labeling_df["labeling_stratum"] == stratum]
        .sort_values("baseline_similarity_score", ascending=False)
        .head(3)
    )


## Итог

Главный артефакт этого ноутбука — `research/dedup/data/labeling_sauces.csv`. После ручной разметки этот файл станет gold-set для сравнения matching engines в следующем ноутбуке.
